# Step 8b: Features 2 and 3, purchase interval and overdue ratio

These describe a customer's **rhythm**, which the model does not have yet:
- **IPI** (purchase interval) = mean days between consecutive purchase days.
- **OverdueRatio** = Recency ÷ IPI. Above 1 means the customer is later than their own usual rhythm.

Customers who bought on only one day have no interval (about a third). We leave them **blank** and let XGBoost handle missing values itself: no fill value, so nothing to leak from train to test (and `Frequency == 1` already flags them).

**Test design (unchanged):** the 7e settings stay FIXED, and each variant is compared with the R, F, M model on the SAME 10 fold splits. Variants: IPI alone, OverdueRatio alone, both. This is also where we compare candidate keep rules on a borderline feature. Code: `src/step8_features.py`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from metrics import r2, drop_top
from step3_split import load_ledger, make_snapshot, TRAIN_CUTOFF
from step7b_xgboost import oof_with_seed
from step8_features import (build_features, screen, summarize, rule_verdicts, customer_bootstrap, make_ratio_tweedie)

pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90)
train = make_snapshot(load_ledger(), TRAIN_CUTOFF)
y = train.target; ya = y.to_numpy()
X_base = build_features(train)
X_both = build_features(train, ("IPI", "OverdueRatio"))
X_both.head()

/data/home/mitanshub/miniconda3/envs/ml_clv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Recency,Frequency,Monetary,IPI,OverdueRatio
Customer ID,,,,,
12346,156,2,169.36,118.0,1.322034
12347,31,1,611.53,NaN,NaN
12348,65,1,222.16,NaN,NaN
12349,34,3,2646.99,91.0,0.373626
12351,2,1,300.93,NaN,NaN


## 1. What the features look like and what they tell us

In [2]:
print(f"IPI missing (one purchase day): {X_both.IPI.isna().sum():,} customers ({X_both.IPI.isna().mean():.0%}); Frequency == 1: {(X_both.Frequency == 1).sum():,}")
display(X_both[["IPI", "OverdueRatio"]].describe(percentiles=[.05, .25, .5, .75, .95]).round(2).T)

scale = X_base["Monetary"].clip(lower=50).to_numpy(); buys = ya > 0
mult = pd.Series(ya[buys] / scale[buys], index=X_both.index[buys])
rows = {}
for c in ("Recency", "Frequency", "Monetary", "IPI", "OverdueRatio"):
    v = X_both.loc[mult.index, c]; ok = v.notna()
    rows[c] = {"customers": int(ok.sum()), "Spearman with real multiplier": v[ok].corr(mult[ok], method="spearman")}
display(pd.DataFrame(rows).T.round(3))

b = pd.cut(X_both.OverdueRatio, [0, .5, 1, 2, 4, 1e9], labels=["<0.5 (early)", "0.5-1", "1-2", "2-4", ">4 (very late)"])
display(pd.Series(buys, index=X_both.index).groupby(b, observed=True).agg(["mean", "size"]).round(2).rename(columns={"mean": "share who buy again", "size": "customers"}).T)

IPI missing (one purchase day): 1,491 customers (35%); Frequency == 1: 1,423


,count,mean,std,min,5%,25%,50%,75%,95%,max
IPI,2751.0,78.48,63.43,1.0,12.76,35.0,60.00,103.83,211.00,364.0
OverdueRatio,2751.0,2.09,11.43,0.0,0.04,0.2,0.48,1.28,5.68,363.0


,customers,Spearman with real multiplier
Recency,2039.0,0.203
Frequency,2039.0,-0.455
Monetary,2039.0,-0.453
IPI,1687.0,0.004
OverdueRatio,1687.0,0.102


OverdueRatio,<0.5 (early),0.5-1,1-2,2-4,>4 (very late)
share who buy again,0.65,0.7,0.59,0.52,0.34
customers,1405.00,514.0,398.00,220.00,214.00


- **IPI alone says nothing about the multiplier** (rank correlation +0.00). Knowing a customer's usual gap does not tell us whether they will spend more or less than their past rate.
- **OverdueRatio does carry a signal about *whether* they come back:** the share who buy again falls from about 65 to 70% (on time or early) to 34% (more than 4 times overdue). But among those who do buy, its link to the multiplier is weak (+0.10).
- The model already has Recency and Frequency, which cover much of this. So expect, at most, a small gain.

## 2. The paired screen: 10 fold splits, settings fixed

In [3]:
variants = {"+ AOV (from 8a)": ("AOV",), "+ IPI": ("IPI",), "+ OverdueRatio": ("OverdueRatio",), "+ IPI + OverdueRatio": ("IPI", "OverdueRatio")}
screens, boots, Xs = {}, {}, {}
for name, extra in variants.items():
    Xs[name] = build_features(train, extra)
    screens[name] = screen(X_base, Xs[name], y)
    boots[name] = customer_bootstrap(X_base, Xs[name], y)
    print(f"=== {name} ===")
    display(summarize(screens[name]).round(4))

=== + AOV (from 8a) ===


,mean,sd,improved,t
R2,-0.0287,0.0113,0.0,-8.0113
MAE gain (GBP),-3.5767,2.6495,1.0,-4.2690
Capture@20%,0.0001,0.0017,5.0,0.1257


=== + IPI ===


,mean,sd,improved,t
R2,-0.0106,0.0112,2.0,-3.0005
MAE gain (GBP),-0.0136,1.5101,4.0,-0.0286
Capture@20%,0.0052,0.0019,10.0,8.7070


=== + OverdueRatio ===


,mean,sd,improved,t
R2,-0.0104,0.0049,0.0,-6.7507
MAE gain (GBP),0.9487,1.0494,7.0,2.8587
Capture@20%,0.0011,0.0016,8.0,2.1140


=== + IPI + OverdueRatio ===


,mean,sd,improved,t
R2,-0.0126,0.0101,0.0,-3.9145
MAE gain (GBP),4.2898,1.7080,10.0,7.9423
Capture@20%,0.0054,0.0017,10.0,10.1560


(`improved` = number of the 10 splits where the new model is better; `t` = mean ÷ (sd ÷ √10).)

Read the "+ IPI + OverdueRatio" block: **MAE and ranking improve in all 10 splits, but R² gets worse in all 10.** The metrics disagree. That is a real borderline feature.

## 3. Why does R² drop? It is the giants

R² is decided by the few biggest customers. Split the squared error into the 10 biggest customers and everybody else, for the baseline and for the model with both features:

In [4]:
top = np.argsort(-ya)[:10]; rest = np.delete(np.arange(len(ya)), top)
rows = []
for s in range(10):
    pb = oof_with_seed(make_ratio_tweedie, X_base, y, s); pn = oof_with_seed(make_ratio_tweedie, X_both, y, s)
    yb, b10 = drop_top(ya, pb, 10); yn, n10 = drop_top(ya, pn, 10)
    rows.append({"R2 diff": r2(ya, pn) - r2(ya, pb), "R2 diff without top 10": r2(yn, n10) - r2(yb, b10),
                 "top-10 sq.err base": ((ya[top] - pb[top]) ** 2).sum() / 1e9, "top-10 sq.err new": ((ya[top] - pn[top]) ** 2).sum() / 1e9,
                 "rest sq.err base": ((ya[rest] - pb[rest]) ** 2).sum() / 1e9, "rest sq.err new": ((ya[rest] - pn[rest]) ** 2).sum() / 1e9})
d = pd.DataFrame(rows); d.index.name = "split"
display(d.round(3))
print(f"top-10 error is higher with the new features in {(d['top-10 sq.err new'] > d['top-10 sq.err base']).sum()}/10 splits; "
      f"error on everyone else is lower in {(d['rest sq.err new'] < d['rest sq.err base']).sum()}/10 splits")
print(f"mean R2 diff {d['R2 diff'].mean():+.4f} | without the top 10: {d['R2 diff without top 10'].mean():+.4f} (better in {(d['R2 diff without top 10'] > 0).sum()}/10 splits)")

,R2 diff,R2 diff without top 10,top-10 sq.err base,top-10 sq.err new,rest sq.err base,rest sq.err new
split,,,,,,
0,-0.005,0.003,6.269,6.511,4.942,4.910
1,-0.007,0.002,7.257,7.608,5.162,5.142
2,-0.009,-0.011,7.551,7.813,4.794,4.926
3,-0.006,-0.007,7.495,7.713,5.128,5.206
4,-0.012,0.007,7.014,7.665,5.026,4.940
5,-0.016,-0.024,6.846,7.293,4.695,4.970
6,-0.007,-0.011,6.373,6.557,4.944,5.068
7,-0.037,-0.006,9.392,11.022,5.589,5.654
8,-0.022,0.007,6.663,7.746,5.053,4.974


top-10 error is higher with the new features in 10/10 splits; error on everyone else is lower in 4/10 splits
mean R2 diff -0.0126 | without the top 10: -0.0042 (better in 4/10 splits)


(Squared errors are in billions of £².) The R² drop comes from the **10 biggest customers**: their error goes up in every split. For the other 4,232 customers the error is slightly lower in most splits. Without the 10 biggest, the R² change is only about -0.004 (better in 4 of 10 splits), i.e. roughly zero. So the extra features make the giants' multipliers slightly noisier, and R² punishes that more than anything.

## 4. How much can we trust the fold-split numbers?

The 10 fold splits **reuse the same customers**, so their spread only measures fold-assignment noise, not "which customers happen to be in our data". A bootstrap over customers measures that second noise (paired, new minus base):

In [5]:
rows = {}
for name, b in boots.items():
    for m, (mean, lo, hi) in b.items():
        rows[(name, m)] = {"mean": mean, "95% low": lo, "95% high": hi, "excludes 0?": "yes" if lo > 0 or hi < 0 else "no"}
display(pd.DataFrame(rows).T.round(4))

mean    95% low   95% high excludes 0?
+ AOV (from 8a)      R2             -0.026451  -0.063162   0.000357          no
                     MAE gain (GBP) -6.090126 -12.609326  -0.121627         yes
                     Capture@20%     0.000966  -0.002911   0.004087          no
+ IPI                R2             -0.002975   -0.01967   0.013727          no
                     MAE gain (GBP)  1.372357  -3.099027    5.39637          no
                     Capture@20%     0.005035  -0.001214   0.011119          no
+ OverdueRatio       R2             -0.006584  -0.020301   0.004079          no
                     MAE gain (GBP)  1.508196  -2.496644   5.667934          no
                     Capture@20%     0.001552  -0.003184   0.006029          no
+ IPI + OverdueRatio R2                -0.005  -0.019562   0.007839          no
                     MAE gain (GBP)  5.729317   1.096532  10.212988         yes
                     Capture@20%     0.005104  -0.001667   0.011982          no

For "+ IPI + OverdueRatio":
- **MAE gain about +£5.7, interval +£1.1 to +£10.2: excludes 0.** A small (about 1%) but real improvement.
- **Capture@20% +0.005, interval -0.002 to +0.012: includes 0.** The "t = 10" from the fold splits was misleading; over customers it is not significant.
- **R² -0.005, interval -0.020 to +0.008: includes 0.** Not significantly worse once customer noise is counted.

## 5. What each candidate keep rule says

| Rule | Idea |
|---|---|
| A | R² better in at least 9 of 10 splits (our first proposal) |
| B | R² better in at least 8 of 10 splits |
| C | mean R² gain > 0 with paired t ≥ 2, and MAE not clearly worse |
| D | over customers: R² *or* MAE significantly better, and neither significantly worse |
| E | D, plus a practical size (MAE gain ≥ £5 or R² gain ≥ +0.005) |

In [6]:
verd = pd.DataFrame({name: rule_verdicts(screens[name], boots[name]) for name in variants})
display(verd.map(lambda v: "KEEP" if v else "reject"))

,+ AOV (from 8a),+ IPI,+ OverdueRatio,+ IPI + OverdueRatio
A: R2 better in >= 9/10 splits,reject,reject,reject,reject
B: R2 better in >= 8/10 splits,reject,reject,reject,reject
"C: R2 mean > 0, t >= 2, MAE not clearly worse",reject,reject,reject,reject
"D: R2 or MAE significantly better over customers, none significantly worse",reject,reject,reject,KEEP
E: D and a practical size (MAE gain >= 5 GBP or R2 >= +0.005),reject,reject,reject,KEEP


## What we learned

Effect of each feature set versus the R, F, M model (bootstrap over customers, fixed 7e settings; **bold** = interval excludes 0):

| Feature set | R² change | MAE gain (£) | Capture@20% change |
|---|---|---|---|
| + AOV (8a) | -0.026 | **-6.1 (worse)** | +0.001 |
| + IPI | -0.003 | +1.4 | +0.005 |
| + OverdueRatio | -0.007 | +1.5 | +0.002 |
| **+ IPI + OverdueRatio** | -0.005 | **+5.7** | +0.005 |

- **Interval and overdue ratio each add almost nothing alone; together they give a small but real MAE gain** (+£5.7, about 1%, interval +£1.1 to +£10.2). Ranking (+0.005) and R² (-0.005) are not significantly different from zero once customer noise is counted.
- **The R² dip is the giants.** The squared error of the 10 biggest customers rises in all 10 splits; for the other 4,232 customers it is slightly lower. Without the top 10, R² changes by only -0.004.
- **The signal is real but small and mostly already in Recency and Frequency:** OverdueRatio separates who comes back (65 to 70% on time vs 34% very late) but barely shifts the multiplier among buyers (+0.10), and IPI alone says nothing about the multiplier (+0.00).
- **The fold-split t-values overstated the evidence** (capture "t = 10" is not significant over customers).

### The keep rule, now with evidence

| Rule | Verdict on the four feature sets |
|---|---|
| A: R² better in ≥ 9/10 splits | rejects all four |
| B: R² better in ≥ 8/10 splits | rejects all four |
| C: R² mean > 0, t ≥ 2, MAE not clearly worse | rejects all four |
| D: R² or MAE significantly better over customers, none significantly worse | keeps only IPI + OverdueRatio |
| E: D plus a practical size (MAE gain ≥ £5 or R² ≥ +0.005) | keeps only IPI + OverdueRatio |

- Rules A to C look only at **R²**, and R² here is decided by about ten customers and by fold assignment. They are blind to a small, real gain in typical-customer error (MAE) and they overstate how certain 10 reused splits are.
- Rules D and E judge each metric by a **bootstrap over customers**, require that nothing gets significantly worse, and (E) a minimum practical size. They are the more honest of the five, and they still reject AOV (its MAE is significantly worse).
- **Caveat:** we tested four variants, and the pair's MAE interval only just excludes 0 (+£1.1 at the low end). When several variants are tried, one can pass by chance. Treat the pair as a **provisional keep**, to be re-checked after all features are in (final re-tune and fresh fold splits).

**Next (step 8c):** spend trend (recent vs earlier months) and return rate (feature window only, by invoices).